# LLM taxonomy cleanup + examples (Ollama via txtai)

Draft notebook inspired by `01_preprocess_taxonomy.ipynb`. It cleans definition/examples with a local LLM and generates short positive/negative examples using parent-path context. For production, move the logic into a Kedro node/pipeline and version outputs.


In [ ]:
import json
import os
import re
from typing import Any, Dict, Optional

import numpy as np
import pandas as pd

from taxomind.utils import embedding_utils
from taxomind.utils.taxonomy_utils import (
    get_parent_chain,
    identify_level,
    process_taxonomy_parent,
)



In [ ]:
TAXONOMY_KEY = "ISIC"

LLM_PROVIDER = "openai"  # "ollama" or "openai"
LLM_MODEL = "gpt-4.1-mini"  # Ollama or OpenAI model name
OLLAMA_URL = "http://localhost:11434"  # Default Ollama API endpoint
OPENAI_BASE_URL = "https://api.openai.com/v1"
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
EMBEDDING_MODEL = "nomic-ai/nomic-embed-text-v2-moe"
EMBEDDING_PREFIX = ""
K_SIMILAR_LABELS = 2
MAX_ROWS = None  # Set to an int for a small dry run
APPLY_CLEANED = False  # Overwrite definition/examples with LLM-cleaned versions


In [ ]:
%load_ext kedro.ipython
%reload_kedro

In [ ]:
if TAXONOMY_KEY == "ISCO":
    dataset_name = "isco_taxonomy_definition"
    column_mapping = {
        "Level": "level",
        "ISCO 08 Code": "code",
        "Title EN": "label",
        "Definition": "definition",
        "Tasks include": "examples",
    }
elif TAXONOMY_KEY == "ISIC":
    dataset_name = "isic_taxonomy_definition"
    column_mapping = {
        "Code": "code",
        "ISIC Rev. 4 label": "label",
        "Inclusions": "examples",
    }


In [ ]:
df = catalog.load(dataset_name)
selected_columns = list(column_mapping.values())
df.rename(columns=column_mapping, inplace=True)
if TAXONOMY_KEY == "ISIC":
    df["definition"] = df["examples"]
    df["level"] = df["code"].apply(identify_level)
    selected_columns += ["definition", "level"]

df = df[selected_columns]
df = process_taxonomy_parent(df)


In [ ]:
REFERENCE_PATTERNS = [
    r"\bSee class\b",
    r"\bSee group\b",
    r"\bSee division\b",
    r"\bSee section\b",
    r"\bSee\b.*\bclass\b",
    r"\bCheck parent\b",
    r"\bCheck\b.*\bparent\b",
    r"\bSee also\b",
]

def _is_missing(value: Any) -> bool:
    if value is None:
        return True
    try:
        missing = pd.isna(value)
    except Exception:
        return False
    if isinstance(missing, bool):
        return bool(missing)
    if hasattr(missing, "shape") and missing.shape == ():
        return bool(missing)
    return False

def _strip_reference_lines(text: Any) -> str:
    if _is_missing(text):
        return ""
    lines = []
    for raw in str(text).splitlines():
        line = raw.strip()
        if not line:
            continue
        if any(re.search(pat, line, flags=re.IGNORECASE) for pat in REFERENCE_PATTERNS):
            continue
        line = re.sub(r"^[\-*\d.\)\(]+\s*", "", line).strip()
        if line:
            lines.append(line)
    return "\n".join(lines).strip()

def _basic_cleanup(text: str) -> str:
    cleaned = _strip_reference_lines(text)
    cleaned = re.sub(r"\s{2,}", " ", cleaned)
    return cleaned.strip()


In [ ]:
code_to_label = df.set_index("code")["label"].to_dict()

def _parent_path_labels(code: str) -> str:
    chain = get_parent_chain(df, code)
    labels = [code_to_label.get(parent_code, parent_code) for parent_code in chain]
    return " > ".join(labels)

df["parent_path"] = df["code"].apply(_parent_path_labels)
df["parent_label"] = df["parentCode"].map(code_to_label).fillna("")

try:
    EMBEDDING_MODEL = context.params.get("model_name", EMBEDDING_MODEL)
    EMBEDDING_PREFIX = context.params.get("embedding_prefix", {}).get("document", EMBEDDING_PREFIX)
except Exception:
    pass

embedding_model = embedding_utils.load_embedding_model(EMBEDDING_MODEL)

def _build_similar_labels(df: pd.DataFrame) -> Dict[str, list]:
    if K_SIMILAR_LABELS <= 0:
        return {code: [] for code in df["code"]}
    similar: Dict[str, list] = {}
    for level, group in df.groupby("level"):
        group_reset = group.reset_index(drop=True)
        labels = group_reset["label"].fillna("").astype(str).tolist()
        codes = group_reset["code"].tolist()
        if len(labels) < 2:
            for code in codes:
                similar[code] = []
            continue
        try:
            vectors, _ = embedding_utils.encode_texts(
                embedding_model,
                labels,
                embed_all=True,
                input_prefix=EMBEDDING_PREFIX,
                batch_size=32,
                show_progress_bar=False,
            )
        except Exception as exc:
            print(f"Embedding failed for level {level}: {exc}")
            for code in codes:
                similar[code] = []
            continue
        sims = vectors @ vectors.T
        for idx, code in enumerate(codes):
            sims[idx, idx] = -1.0
            top_idx = np.argsort(sims[idx])[::-1][:K_SIMILAR_LABELS]
            candidates = []
            for j in top_idx:
                candidates.append({
                    "label": labels[j],
                    "definition": _basic_cleanup(group_reset.at[j, "definition"]),
                    "examples": _basic_cleanup(group_reset.at[j, "examples"]),
                })
            similar[code] = candidates
    return similar

similar_label_map = _build_similar_labels(df)
df["similar_labels"] = df["code"].map(similar_label_map).apply(lambda value: value or [])



In [ ]:
import requests

try:
    from txtai.pipeline import LLM
except Exception:
    LLM = None

try:
    from dotenv import load_dotenv

    load_dotenv()
except Exception:
    pass

_llm = None
if LLM_PROVIDER == "ollama" and LLM is not None:
    _llm = LLM(f"ollama:{LLM_MODEL}", url=OLLAMA_URL)

def _run_ollama(prompt: str) -> str:
    response = requests.post(
        f"{OLLAMA_URL}/api/chat",
        json={
            "model": LLM_MODEL,
            "stream": False,
            "messages": [{"role": "user", "content": prompt}],
        },
        timeout=120,
    )
    response.raise_for_status()
    payload = response.json()
    content = payload.get("message", {}).get("content") or payload.get("response", "")
    return content or ""

def _run_openai(prompt: str) -> str:
    if not OPENAI_API_KEY:
        raise ValueError("OPENAI_API_KEY is not set")
    response = requests.post(
        f"{OPENAI_BASE_URL}/chat/completions",
        headers={"Authorization": f"Bearer {OPENAI_API_KEY}"},
        json={
            "model": LLM_MODEL,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0,
        },
        timeout=120,
    )
    response.raise_for_status()
    payload = response.json()
    return (
        payload.get("choices", [{}])[0]
        .get("message", {})
        .get("content", "")
        .strip()
    )

def run_llm(prompt: str) -> str:
    if LLM_PROVIDER == "ollama":
        if _llm is not None:
            return _llm(prompt)
        return _run_ollama(prompt)
    if LLM_PROVIDER == "openai":
        return _run_openai(prompt)
    raise ValueError(f"Unsupported LLM_PROVIDER: {LLM_PROVIDER}")


In [ ]:
def _parse_json_block(text: str) -> Optional[Dict[str, Any]]:
    if text is None:
        return None
    payload = str(text).strip()
    candidates = [payload]
    if "```" in payload:
        parts = payload.split("```")
        for idx in range(1, len(parts), 2):
            block = parts[idx].strip()
            if block.startswith("json"):
                block = block[4:].strip()
            candidates.append(block)
    for candidate in candidates:
        try:
            parsed = json.loads(candidate)
        except json.JSONDecodeError:
            continue
        if isinstance(parsed, dict):
            return parsed
    return None

def _normalize_multiline(value: Any, max_lines: int) -> str:
    if value is None:
        return ""
    if isinstance(value, list):
        lines = [str(item).strip() for item in value if str(item).strip()]
    else:
        lines = [line.strip() for line in str(value).splitlines() if line.strip()]
    cleaned = []
    for line in lines:
        line = re.sub(r"^[\-*\d.\)\(]+\s*", "", line).strip()
        line = line.replace("\u2022", "").strip()
        if line:
            cleaned.append(line)
    if max_lines is not None:
        cleaned = cleaned[:max_lines]
    return "\n".join(cleaned)


In [ ]:
def _format_similar_labels(similar: Any) -> str:
    if not similar:
        return "(none)"
    blocks = []
    for item in similar:
        label = item.get("label", "")
        definition = item.get("definition", "")
        examples = item.get("examples", "")
        blocks.append(
            f"- Label: {label}\n  Definition: {definition}\n  Examples: {examples}".strip()
        )
    return "\n".join(blocks)

def _build_prompt(row: pd.Series) -> str:
    parent_path = row.get("parent_path") or "(root)"
    definition = row.get("definition") or ""
    examples = row.get("examples") or ""
    similar_labels = _format_similar_labels(row.get("similar_labels") or [])

    return f"""You clean taxonomy text and craft short query examples.

Label: {row.get('label')}
Level: {row.get('level')}
Parent path labels: {parent_path}

Definition (raw):
{definition}

Examples (raw):
{examples}

Similar labels at the same level (use for negatives only):
{similar_labels}

Rules:
- Remove cross references like "See class" or "Check parent label"
- Keep meaning faithful to label and parent path
- Use the same language as the label/definition when present
- definition_clean: short description if empty or not descriptive enough
- examples_clean: 2-4 short examples if there are usable examples; if none, return empty string (do not hallucinate)
- positive_examples: 2 short query phrases (1-4 words) as if someone were casually describing it, newline separated
- negative_examples: 2 short query phrases as if someone were casually describing it but plausible and wrong, newline separated
- Base negative_examples on the similar labels when available
- Do not include the label or code in the examples
- If raw text is empty, infer cautiously from label and parent path only

Return JSON with keys: definition_clean, examples_clean, positive_examples, negative_examples.
"""


In [ ]:
def llm_enrich_row(row: pd.Series) -> Dict[str, str]:
    prompt = _build_prompt(row)
    response = run_llm(prompt)
    parsed = _parse_json_block(response)
    if not parsed:
        return {
            "definition_clean": _basic_cleanup(row.get("definition")),
            "examples_clean": _basic_cleanup(row.get("examples")),
            "positive_examples": "",
            "negative_examples": "",
        }
    return {
        "definition_clean": _basic_cleanup(parsed.get("definition_clean")),
        "examples_clean": _normalize_multiline(parsed.get("examples_clean"), max_lines=4),
        "positive_examples": _normalize_multiline(parsed.get("positive_examples"), max_lines=2),
        "negative_examples": _normalize_multiline(parsed.get("negative_examples"), max_lines=2),
    }


In [ ]:
rows = df.copy()
if MAX_ROWS:
    rows = rows.head(MAX_ROWS)

try:
    from tqdm.auto import tqdm

    iterator = tqdm(rows.iterrows(), total=len(rows))
except Exception:
    iterator = rows.iterrows()

results = []
for _, row in iterator:
    payload = llm_enrich_row(row)
    payload["code"] = row["code"]
    results.append(payload)

results_df = pd.DataFrame(results).set_index("code")
df_llm = df.set_index("code").join(results_df).reset_index()

if "similar_labels" in df_llm.columns:
    df_llm = df_llm.drop(columns=["similar_labels"])

if APPLY_CLEANED:
    df_llm["definition"] = df_llm["definition_clean"]
    df_llm["examples"] = df_llm["examples_clean"]


To persist results, save via Kedro. If you want a separate dataset, add it in `conf/base/catalog.yml` first.


In [ ]:

catalog.save("taxonomy_definition_llm", {TAXONOMY_KEY: df_llm})


In [ ]:
TAXONOMY_KEY = "ISCO"
df = catalog.load("taxonomy_definition_llm")[TAXONOMY_KEY]()
df.loc[df["parentCode"] == "__root__", "parentCode"] = None
df.drop(columns=['id'], inplace=True, errors='ignore')
df.rename(columns={'parentCode': 'parent_id'}, inplace=True)
df.rename(columns={'code': 'id'}, inplace=True)
df['definition'] = df['definition_clean'].copy()
df['examples'] = df['examples_clean'].copy()

In [ ]:
columns_to_drop = ['code', 'isLeaf', 'parent_path', 'parent_label', 'definition_clean', 'examples_clean', 'positive_examples', 'negative_examples', 'taxonomyKey']
columns_to_keep = [col for col in df.columns if col not in columns_to_drop]
df = df[columns_to_keep].copy()

catalog.save("taxonomy_definition_for_testing", {TAXONOMY_KEY: df})